In [1]:
#Import libraries

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

In [2]:
#Import relavent data with standardised IDs

H_P2 = pd.read_csv("Data//Standardised_IDs//HarrisPartII.csv")         
V_T2 = pd.read_csv("Data//Standardised_IDs//vandenBerg_table2.csv")        
K_21 = pd.read_csv("Data//Standardised_IDs//Krause21.csv")      

In [3]:
# Cleanup Metalicities

Harris_Fe = H_P2[["ID","wt","[Fe/H]"]]  # |-> Keeps only IDs, metality values and metalicity value weights from imported data
VanBerg_Fe = V_T2[["ID","FeH"]]         # |
Krause_Fe = K_21[["ID","FeH"]]          # |

VK_Fe = pd.merge(VanBerg_Fe,Krause_Fe,on="ID",how="outer")  # |-> Merges imported data into a single DataFrame
HVK_Fe = pd.merge(Harris_Fe,VK_Fe,on="ID",how="outer")      # |    -> Dataframe called HVK because it contains data from Harris, VandenBerg and Krause
HVK_Fe.rename(columns={                                 # | -> Renames columns to reflect the source of the data
    "wt"    : "wt_H",                                   # |
    "[Fe/H]": "[Fe/H]_H",                               # |     -> "H" : "Harris"
    "FeH_x" : "[Fe/H]_V",                               # |     -> "V" : "VandenBerg"
    "FeH_y" : "[Fe/H]_K"                                # |     -> "K" : "Krause"
    }, inplace=True)
HVK_Fe = HVK_Fe.replace(r'^\s*$', np.nan)       # | -> Replace whitespace and empty with NaN
HVK_Fe = HVK_Fe.replace(r'     ', np.nan)       # |

HVK_Fe                                          # | -> Display Result

,ID,wt_H,[Fe/H]_H,[Fe/H]_V,[Fe/H]_K
0,NGC104,10.0,-0.72,-0.76,-0.76
1,NGC288,3.0,-1.32,-1.32,-1.32
2,NGC362,5.0,-1.26,-1.30,-1.26
3,Whiting1,1.0,-0.7,NaN,NaN
4,NGC1261,3.0,-1.27,-1.27,-1.08
...,...,...,...,...,...
153,NGC7099,4.0,-2.27,-2.33,-2.33
154,Pal12,6.0,-0.85,-0.81,-0.81
155,Pal13,2.0,-1.88,NaN,NaN
156,NGC7492,5.0,-1.78,NaN,NaN


In [4]:
# Merge metalicity data using weighted average of metalicity data

def MetalicityWeightedAverage(FeData=HVK_Fe):
    DF = pd.DataFrame() # Creates a fresh dataframe to write to as to avoid modifying the original data
    DF[["[Fe/H]_H","[Fe/H]_V","[Fe/H]_K","wt_H"]] = FeData[["[Fe/H]_H","[Fe/H]_V","[Fe/H]_K","wt_H"]]

    NaNDict = {"NA" : 0} # Will be used later to remove marked datapoints

    DF[["[Fe/H]_V","[Fe/H]_K"]] = DF[["[Fe/H]_V","[Fe/H]_K"]].fillna("NA")  # Goes through VandedBerg and Krause and marks all NaN values (where we dont have data) 
    DF["wt_V"] = (DF["[Fe/H]_V"].map(NaNDict)).fillna(1.0)        # Sets weights for metalicities we dont have to 0 and the rest to 1 (as a placeholder)
    DF["wt_K"] = (DF["[Fe/H]_K"].map(NaNDict)).fillna(1.0)        # Repeats but for Krause data
    DF = DF.fillna(0.0)                                           # Repeats but for Harris data (here we just replace NaN to 0)
    DF = DF.replace('NA', 0)                                      # Replaces the "NA" mark with zero so that we can do dot products


    H_weighted = np.array(DF["wt_H"],dtype="f")*np.array(DF["[Fe/H]_H"],dtype="f")
    V_weighted = np.array(DF["wt_V"],dtype="f")*np.array(DF["[Fe/H]_V"],dtype="f")
    K_weighted = np.array(DF["wt_K"],dtype="f")*np.array(DF["[Fe/H]_K"],dtype="f")

    WeightedSum = (H_weighted+V_weighted+K_weighted)
    SumWeights = np.array(DF["wt_H"],dtype="f")+np.array(DF["wt_V"],dtype="f")+np.array(DF["wt_K"],dtype="f")

    with np.errstate(invalid="ignore"):
        WeightedAvg = np.divide(WeightedSum, SumWeights)

    return WeightedAvg

In [5]:
# Merge metalicity data using agreement from different sources

def MetalicityAgreement(AllFeH=HVK_Fe,Default="[Fe/H]_H"):
    #print(AllFeH[Default])
    FeH_Cols = []
    for i in np.array(AllFeH.columns,dtype="str"):
        if i[:7]=="[Fe/H]_":
            FeH_Cols = np.append(FeH_Cols,i)
    
    FeH_Data = AllFeH[FeH_Cols].astype("f")#.fillna(np.inf)
    
    
    def UniqueCounts(Series):
        return np.unique(Series,return_counts=True,return_inverse=True)
    ValuesCountsInverse = FeH_Data.apply(UniqueCounts,axis=1)

    def AgreedValue(ValuesInverseCounts):
        Values, Inv, Counts = ValuesInverseCounts
        #print(Values,Counts,Inv)
        MaxCounts = max(Counts)
        HighestCountValue = Values[Counts==MaxCounts]
        SecondHighestCountValue = Values[Counts==MaxCounts-1]
    
        if (str(HighestCountValue[0])!="nan"):
            MostPopular = HighestCountValue
        else: MostPopular = SecondHighestCountValue
        
        #print(HighestCountValue,MostPopular)
        return MostPopular

    AgreedValueSeries = ValuesCountsInverse.apply(AgreedValue)
    FeH_Data["AgreedValue"]=AgreedValueSeries

    #print(FeH_Data)

    def ReplaceNoAgreementWithDefault(FeH_Data,Default=Default):
        AgreedValue = FeH_Data["AgreedValue"]        
        if np.size(AgreedValue)==1:
            return AgreedValue[0]
        else:
            return FeH_Data[Default]

    FinalValueSeries = FeH_Data.apply(ReplaceNoAgreementWithDefault,axis=1)
        
    return FinalValueSeries

In [6]:
def AgreementAllDefaults(AllFeHData=HVK_Fe):
    FeH_Cols = []
    SuffixList = []
    for i in np.array(AllFeHData.columns,dtype="str"):
        if i[:7]=="[Fe/H]_":
            FeH_Cols = np.append(FeH_Cols,i)
            Suffix = "_%s"%i[-1:]
            SuffixList = np.append(SuffixList,Suffix)
    
    DF = pd.DataFrame()
    DF["ID"] = AllFeHData["ID"]

    for i in range(len(FeH_Cols)):
        Default = FeH_Cols[i]
        Suffix = SuffixList[i]
        DF["[Fe_H]_Agr%s"%Suffix] = MetalicityAgreement(AllFeH=HVK_Fe,Default=Default)
    
    return DF

In [7]:
FeH_Agr = AgreementAllDefaults(HVK_Fe)                          # Creates column of merged metalicity data by agreement for each possible "default" data source
FeH_Agr["[FeH]_WtdAvg"] = MetalicityWeightedAverage(HVK_Fe)     # Adds column of merged metalicity data by weighted average

MergedMetalicities = pd.merge(FeH_Agr,HVK_Fe,on="ID")           # Adds merged data to the original data to help with possible future cross-referencing and/or troubleshooting

def Rename2LongName():
    NameDict={
        "H" :   "Harris",
        "V" :   "vandenBerg",
        "K" :   "Krause"}

    for column in MergedMetalicities.columns:
        if column[-1] in (list(NameDict.keys())):
            MergedMetalicities.rename(columns={column:str(column[:-1]+NameDict[column[-1]])}, inplace=True)

Rename2LongName()                   # Renames Columns to use longer version of data source names. Comment out and rerun if unwanted.
MergedMetalicities                  # Displays DataFrame

,ID,[Fe_H]_Agr_Harris,[Fe_H]_Agr_vandenBerg,[Fe_H]_Agr_Krause,[FeH]_WtdAvg,wt_Harris,[Fe/H]_Harris,[Fe/H]_vandenBerg,[Fe/H]_Krause
0,NGC104,-0.76,-0.76,-0.76,-0.726667,10.0,-0.72,-0.76,-0.76
1,NGC288,-1.32,-1.32,-1.32,-1.320000,3.0,-1.32,-1.32,-1.32
2,NGC362,-1.26,-1.26,-1.26,-1.265714,5.0,-1.26,-1.30,-1.26
3,Whiting1,-0.70,-0.70,-0.70,-0.700000,1.0,-0.7,NaN,NaN
4,NGC1261,-1.27,-1.27,-1.27,-1.232000,3.0,-1.27,-1.27,-1.08
...,...,...,...,...,...,...,...,...,...
153,NGC7099,-2.33,-2.33,-2.33,-2.290000,4.0,-2.27,-2.33,-2.33
154,Pal12,-0.81,-0.81,-0.81,-0.840000,6.0,-0.85,-0.81,-0.81
155,Pal13,-1.88,-1.88,-1.88,-1.880000,2.0,-1.88,NaN,NaN
156,NGC7492,-1.78,-1.78,-1.78,-1.780000,5.0,-1.78,NaN,NaN


In [8]:
# Thingy to save to Directory:
DirName = "MetalicityMerge"
FileName = "MergedMetalicities"

if not os.path.exists("Data//%s"%DirName):   # Check if the folder has been created
    os.makedirs("Data//%s"%DirName)          # If it hasnt been made yet, create it

path = "Data//"+str(DirName)+"//"+str(FileName)+".csv"
if os.path.exists(path):
    yn = input("The file '%s' already exists. Overwrite [y/n]?"%FileName)
    if yn == "y":
        MergedMetalicities.to_csv(path)
else:   MergedMetalicities.to_csv(path)